# Week 10 — Deep Probabilistic Models and Simulation-Based Inference

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../11_DL/01_DL_Intro.ipynb`
- `../../11_DL/02_DL_Regression.ipynb`
- `../../14_SBI/SBI_short.ipynb`
- `../../14_SBI/SBI_full.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Explain conditional density estimation and why point regression is insufficient for degenerate inverse problems.
- Describe normalizing flows conceptually.
- Distinguish neural posterior, likelihood and likelihood-ratio estimation.
- Design simulation-based calibration and posterior predictive checks.
- Recognise prior support, identifiability and amortisation as central SBI issues.

## 1. Why a posterior rather than a point prediction?


Inverse problems can be many-to-one: several physical parameter combinations may produce similar observables. A regressor trained with squared error tends toward a conditional mean, which can lie in a region that is not itself physically plausible.

The target of Bayesian inverse inference is a distribution,

\[
p(\theta\mid x),
\]

not just \(\hat\theta(x)\). A conditional density can represent skewness, multimodality and parameter degeneracy.

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — A toy likelihood-free simulator

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(2026)

def simulator(theta, n=50):
    # nonlinear, noisy toy observable
    return rng.normal(np.sin(theta),0.25,size=n).mean()

theta_train=rng.uniform(-np.pi,np.pi,20000)
x_train=np.array([simulator(t) for t in theta_train])
plt.scatter(theta_train[:1500],x_train[:1500],s=4,alpha=.3)
plt.xlabel('theta'); plt.ylabel('summary x'); plt.show()

## 2. Conditional density estimation and normalizing flows


A normalizing flow starts with a simple random variable \(z\sim p_Z(z)\) and applies an invertible transformation \(\theta=f_\phi(z;x)\). The change-of-variables formula gives

\[
p_\phi(\theta\mid x)
=
p_Z(f_\phi^{-1}(\theta;x))
\left|\det\frac{\partial f_\phi^{-1}}{\partial\theta}\right|.
\]

A sequence of invertible transformations can represent flexible densities while retaining tractable likelihood evaluation. In SBI, the conditioning variable \(x\) can be a vector of summary statistics or a learned representation of the raw observation.

### Worked computational demonstration — ABC approximation to illustrate degeneracy

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
x_obs=0.7
eps=.03
keep=np.abs(x_train-x_obs)<eps
abc=theta_train[keep]
print('accepted',keep.sum())
plt.hist(abc,bins=50,density=True)
plt.xlabel('theta | x≈0.7'); plt.show()
# The multimodality motivates conditional density estimation.

## 3. When the likelihood is unavailable but simulation is easy


Suppose we can draw

\[
\theta\sim p(\theta),\qquad x\sim p(x\mid\theta)
\]

but cannot evaluate \(p(x\mid\theta)\). Simulation-based inference learns components needed for posterior inference from many simulated pairs \((\theta,x)\).

Common strategies are:

- neural posterior estimation (NPE): learn \(p(\theta\mid x)\);
- neural likelihood estimation (NLE): learn \(p(x\mid\theta)\);
- neural ratio estimation (NRE): learn a likelihood-to-evidence or related density ratio.

These methods shift computation toward simulation and training, often enabling fast amortized inference for many sources later.

### Worked computational demonstration — Coverage calculation pattern

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
def interval_coverage(truth, lo, hi):
    truth=np.asarray(truth); lo=np.asarray(lo); hi=np.asarray(hi)
    return np.mean((truth>=lo)&(truth<=hi))

# For your SBI validation set:
# cov68 = interval_coverage(theta_true, q16, q84)
# cov95 = interval_coverage(theta_true, q025, q975)

## 4. The simulation prior defines what can be learned


The training simulations cover a parameter region determined by the simulation prior. An amortized model cannot reliably infer parameters in regions it has never seen. A posterior restricted by simulation support may look precise while being structurally incapable of representing the real source.

This is a particularly important lesson for the binary-SMBH catalogue: the distribution of total mass, \(q\), period and eccentricity is part of the training design. Students must distinguish learning the inverse mapping under that design from learning a universal physical law.

## 5. Identifiability and summary statistics


Even with infinite simulations, \(\theta\) is not identifiable if the observables contain insufficient information. Before training a large SBI network, explore whether the proposed target leaves measurable signatures in light-curve features.

For the final project, \(\log P\) is likely to leave time-domain information, whereas the identifiability of \(q\) from simple summary features must be piloted. An uninformative broad posterior can be the *correct* scientific result.

Hand-crafted summaries reduce dimensionality but can discard information. Learned encoders can retain more, but introduce extra modelling and validation requirements.

## 6. Simulation-based calibration


For SBC, repeatedly draw

\[
\theta^\ast\sim p(\theta),\qquad
x^\ast\sim p(x\mid\theta^\ast),
\]

infer the posterior \(p(\theta\mid x^\ast)\), and record the rank of \(\theta^\ast\) among posterior samples. Under a calibrated inference algorithm and the assumed generative model, ranks should be approximately uniform.

Coverage tests are a related, accessible diagnostic for MSc work. If 68% credible intervals contain truth only 45% of the time, the method is overconfident even if RMSE is small.

## 7. Posterior predictive validation


Inference should close the generative loop. Draw \(\theta\) from the inferred posterior, generate replicated observables \(x_{\rm rep}\), and compare them with the observed \(x\).

This asks whether the inferred physical explanation can regenerate the data. It is especially important when neural density estimators can produce plausible numerical posteriors even under model misspecification.

## Deep dive — an end-to-end SBI workflow

A defensible SBI project can be organised as:

1. define the scientific parameter vector \(\theta\);
2. define the observation \(x\);
3. specify the simulation prior;
4. generate/organise training simulations;
5. split by independent simulation unit;
6. choose summaries or an encoder;
7. train the density estimator;
8. validate on held-out simulations;
9. perform SBC/coverage checks;
10. perform posterior predictive checks;
11. stress-test simulator misspecification.

The simulation budget is a resource. More simulations do not automatically help if they cover irrelevant parameter volume. Sequential SBI strategies can focus simulations where the observed data have support, but must be used carefully to preserve calibration.

### Amortisation

If a network learns \(p_\phi(\theta\mid x)\) across the simulation distribution, inference for a new source is fast. The expensive step is paid during training. This is attractive for LSST-scale catalogues, but it means a systematic training flaw can propagate to millions of posterior estimates.

### Diagnostics before architecture

Before selecting a flow depth or transformer size, plot target-feature relationships, nearest-neighbour distances, simple regression performance and posterior widths. If the data contain little information about a target, a more expressive neural density estimator should represent broad uncertainty—not manufacture precision.

The correct result of SBI can be "the posterior is close to the prior."

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Crete `11_DL` and `14_SBI` notebooks.
- Cranmer, Brehmer & Louppe, simulation-based inference review.
- Talts et al., simulation-based calibration.

# Part II — Extended lecture development

## 16. Neural-network optimisation details

Gradient descent updates parameters

\[
\phi_{t+1}=\phi_t-\eta\nabla_\phi L(\phi_t).
\]

Mini-batch stochastic gradients trade noisy estimates for computational
efficiency. Momentum accumulates a velocity-like average; adaptive optimisers
such as Adam rescale updates using running first/second moments.

Training curves should show both training and validation loss. Divergence
between them is evidence of overfitting, not a reason simply to train longer.

## 17. Regularisation

Common regularisation strategies include:
- weight decay;
- dropout;
- early stopping;
- data augmentation;
- smaller architecture;
- physically motivated constraints.

Regularisation improves generalisation only relative to a validation regime.
It cannot guarantee robustness outside the training distribution.

## 18. Conditional density estimation as supervised learning

NPE training can be viewed as supervised conditional density estimation. Given
pairs \((x_i,\theta_i)\), optimise

\[
-\sum_i\log q_\phi(\theta_i\mid x_i).
\]

This is analogous to regression but with a full density loss rather than MSE.

The quality of the learned posterior depends on:
- simulation coverage;
- density-estimator capacity;
- optimisation;
- conditioning representation;
- sample size;
- validation.

## 19. Change of variables in normalizing flows

For invertible \(z=f^{-1}(\theta)\),

\[
\log p_\theta(\theta)
=
\log p_z(z)
+
\log\left|
\det\frac{\partial z}{\partial\theta}
\right|.
\]

A flow composes transformations whose Jacobian determinants are tractable.
Autoregressive flows achieve flexibility while preserving tractable
determinants.

The key conceptual benefit is exact normalised density evaluation, unlike
implicit generators.

## 20. ABC and the curse of dimensionality

If summaries have dimension \(d\), requiring all of them to lie within a small
tolerance ball becomes exponentially difficult as \(d\) grows.

Acceptance rate collapses, forcing either huge simulation budgets or a large
tolerance that blurs the posterior.

This motivates learned summaries and neural density/ration estimators.

## 21. Sequential SBI

Instead of simulating everywhere under a broad prior, sequential methods use an
initial inference to focus later simulations near regions relevant to the
observation.

This can drastically improve simulation efficiency. But proposal correction and
calibration become important because later simulations no longer follow the
original prior.

For an introductory Master's course, amortised prior-wide inference is easier
to validate; sequential methods are a frontier extension.

## 22. Coverage conditional on parameter space

Global 68% coverage can hide failure near boundaries. Evaluate coverage in bins
of \(q\), \(\log P\), redshift or S/N.

A useful table is:

| region | N | bias | RMSE | 68% coverage | 95% coverage | median width |

The scientific question is not only "is the network calibrated on average?"
but "where does it become overconfident?"

## 23. SBI and nuisance parameters

Suppose \(\theta\) contains targets while \(\eta\) contains nuisance quantities.
Simulations draw both:

\[
\theta,\eta \sim p(\theta,\eta),
\qquad
x\sim p(x\mid\theta,\eta).
\]

An NPE trained for \(\theta\) implicitly marginalises nuisance variation
according to the simulation distribution.

This is powerful, but it means an incorrect nuisance distribution in simulations
can bias the learned posterior.

## 24. Posterior collapse toward the simulation prior

If \(x\) contains little information about a target, Bayes gives

\[
p(\theta\mid x)\approx p(\theta).
\]

This is not a failure of inference. It is the correct statement of
non-identifiability under the model.

A network that instead returns narrow posteriors is overconfident and should
fail coverage tests.

This is the key pilot question for mass ratio \(q\) in the final project.

In [ ]:
# Prior-vs-posterior width thought experiment
prior = rng.uniform(.1,1.0,100_000)
print("q prior 68% width:",
      np.quantile(prior,.84)-np.quantile(prior,.16))
print("If a posterior has nearly the same width for every object,")
print("the observable features may carry little q information.")

## 25. Likelihood-free does not mean assumption-free

SBI replaces explicit likelihood evaluation with simulations, but still assumes:
- the simulator family is adequate;
- parameter support is relevant;
- nuisance distributions are appropriate;
- training converges;
- representation retains information.

The phrase "likelihood-free" must never be interpreted as "model-free."

## 26. A rigorous SBI result

Report:
- simulator/data provenance;
- targets and support;
- observables/summaries;
- architecture and training split;
- baseline;
- coverage/SBC;
- posterior predictive checks;
- OOD/domain-shift tests;
- failure cases;
- simulation-to-real scope.

A beautiful corner plot without calibration is not sufficient.

# Part III — Worked SBI problems

## Problem A — posterior equals prior

You train an NPE for \(q\). For almost every object,

\[
q_\phi(q\mid x)\approx p(q).
\]

Possible interpretation: the supplied observables contain little information
about \(q\) after nuisance variation. This is an identifiability result, not
necessarily a training failure.

Check:
- validation log density/coverage;
- simple predictive baselines;
- dependence between \(q\) and observable summaries;
- whether posterior predictive simulations still match the observation.

## Problem B — narrow but miscalibrated posterior

A network gives very narrow 68% intervals but only 35% contain truth.

The model is overconfident.

Improving RMSE without fixing coverage does not solve the probabilistic
inference problem. Calibration is part of correctness when the output is a
posterior approximation.

## Problem C — prior support boundary

Training simulations cover \(P\in[0.01,10]\) years. A real source may have a
true period of 15 years.

An amortised posterior cannot be validated there. It may pile probability at
the boundary or return an apparently plausible in-support value.

The correct response is not extrapolation confidence; it is to flag support
failure and, if scientifically justified, extend the simulation campaign.

# Part IV — Additional SBI seminar

## Posterior diagnostics must be simulation diagnostics

For ordinary MCMC, chain convergence diagnostics test computation conditional on
the posterior target. SBI adds a second layer: even a perfectly optimised neural
network may approximate the wrong posterior.

Therefore distinguish:

1. **training diagnostics** — loss, validation loss, optimisation stability;
2. **inference calibration** — SBC/coverage on held-out simulations;
3. **model adequacy** — posterior predictive checks;
4. **domain validity** — whether the simulator adequately spans deployment.

A low neural validation loss is only item 1.

## Simulation efficiency versus coverage

If simulations are expensive, one might focus training on regions near an
observation. This improves efficiency but complicates calibration and
amortisation.

The course uses the supplied large simulation set, so a simple held-out
prior-wide design is pedagogically preferable: students can clearly separate
training, validation and blind test without sequential-proposal corrections.